In [1]:
import os
import time
import pandas as pd

from pathlib import Path
from PIL import Image

env = os.environ.copy()
env['LD_LIBRARY_PATH'] = '/usr/local/lib:' + env.get('LD_LIBRARY_PATH', '')
env['PATH'] = '/usr/local/bin:' + env.get('PATH', '')

In [2]:
import subprocess

def compressImg(input_path, output_path, distance=0, effort=7):
    """
    Compress an image using JPEG XL with specified fidelity and effort.

    Args:
        distance (float): Compression distance for visual fidelity (0 = lossless).
        effort (int): Effort level for encoding (1 = fastest, 9 = slowest, default 7).
    """
    # Build the command
    cmd = ['cjxl', str(input_path), str(output_path), '--distance', str(distance), '--effort', str(effort)]
    try:
        subprocess.run(cmd, check=True, env=env)
    except Exception as e:
        print(f'Error compressing {input_path}: {e}.')
        return False
    return True



def decompressImg(input_path, output_path):
    """
    DeCompress an JPEG-XL img.
    """
    cmd = ['djxl', str(input_path), str(output_path)]
    try:
        subprocess.run(cmd, check=True, env=env)
    except Exception as e:
        print(f'Error decompressing {input_path}: {e}.')
        return False
    return True

In [ ]:
def cal_compression_ratio(original_path, compressed_path):
    """
    Calculate compression rate based on file sizes.
    """
    original_size = os.path.getsize(original_path)
    compressed_size = os.path.getsize(compressed_path)

    with Image.open(original_path) as img:
        width, height = img.size
        npixels = width * height
    bpsp = (compressed_size * 8) / (npixels * 3)
    compression_ratio = compressed_size / original_size
    
    return compression_ratio, original_size, compressed_size, bpsp, width, height, npixels

In [4]:
def process_dataset(dataset_path, output_dir, mode="compress", cal_metric=True, verbose=False):
    """
    Process a dataset (Kodak or CLIC) using JPEG-XL, with optional compression rate calculation.
    """
    dataset_path = Path(dataset_path)
    output_dir   = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    imgs = list(dataset_path.glob('*.png'))
    if len(imgs) == 0:
        print(f'No Imgs found in {dataset_path}.')
        return
    
    rows = []
    for img in imgs:
        output_path = output_dir / f'{img.stem}.jxl' if mode == 'compress' else output_dir / f'{img.stem}_recon.png'
        
        start = time.time()
        if mode == 'compress':
            success = compressImg(img, output_path)
        elif mode == 'decompress':
            success = decompressImg(img, output_path)
        elapsed = time.time() - start   
          
        if success and cal_metric and mode == 'compress':
            compression_ratio, original_size, compressed_size, bpsp, width, height, npixels = cal_compression_ratio(img, output_path)
            if verbose:
                print(f'{img.name}: compression ratio = {compression_ratio:.2%} (Original: {original_size} bytes, Compressed: {compressed_size} bytes), bpsp: {bpsp}.')
            rows.append([img.name, bpsp, compression_ratio, original_size, compressed_size, width, height, npixels, elapsed])
            
    return rows

In [5]:
""" 1. compress Touch and Go (png) """
dataset_path = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/compressed/jxl'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/jxl_TouchandGoDataset-v2.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/jxl_TouchandGoDataset-v2.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,1400.000000,1400.000000,1400.000000,1400.000000,1400.0,1400.0,1400.0,1400.000000,1400.000000
mean,0.738656,0.292838,288034.526429,85093.217143,640.0,480.0,307200.0,0.422156,2.500300
std,0.168779,0.024207,44205.507025,19443.335851,0.0,0.0,0.0,0.036184,0.383728
min,0.358924,0.214965,192348.000000,41348.000000,640.0,480.0,307200.0,0.323046,1.669687
25%,0.623917,0.275976,253517.500000,71875.250000,640.0,480.0,307200.0,0.393931,2.200673
50%,0.704288,0.289100,276483.000000,81134.000000,640.0,480.0,307200.0,0.421696,2.400026
75%,0.784991,0.307197,310647.000000,90431.000000,640.0,480.0,307200.0,0.445104,2.696589
max,1.524054,0.380399,481097.000000,175571.000000,640.0,480.0,307200.0,0.559108,4.176189


In [6]:
""" 2. compress Objectfolder (png) """
dataset_path = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/compressed/jxl'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/jxl_ObjectFolder_1.0.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/jxl_ObjectFolder_1.0.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.0,20000.0,20000.0,20000.000000,20000.000000
mean,3.656669,0.920646,28541.906000,26328.013700,160.0,120.0,19200.0,0.081908,3.964154
std,0.427596,0.021998,2695.165442,3078.688188,0.0,0.0,0.0,0.011593,0.374329
min,2.928194,0.868205,24187.000000,21083.000000,160.0,120.0,19200.0,0.057748,3.359306
25%,3.329722,0.904262,26490.000000,23974.000000,160.0,120.0,19200.0,0.074039,3.679167
50%,3.519861,0.913965,27698.000000,25343.000000,160.0,120.0,19200.0,0.079797,3.846944
75%,3.927118,0.936808,30118.000000,28275.250000,160.0,120.0,19200.0,0.087653,4.183056
max,5.803194,1.001201,41962.000000,41783.000000,160.0,120.0,19200.0,0.135508,5.828056


In [7]:
""" 3. compress SSVTP (png) """
dataset_path = '/data/ssd/zhaoy/datasets/SSVTP/dataset-comp/test'
output_dir = '/data/ssd/zhaoy/datasets/SSVTP/compressed/jxl'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/jxl_SSVTP.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/jxl_SSVTP.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,918.000000,918.000000,918.000000,918.000000,918.0,918.0,918.0,918.000000,918.000000
mean,1.477662,0.659590,64301.678649,42556.651416,240.0,320.0,76800.0,0.188850,2.232697
std,0.181336,0.032525,4853.413152,5222.471979,0.0,0.0,0.0,0.028519,0.168521
min,1.157049,0.596334,53899.000000,33323.000000,240.0,320.0,76800.0,0.144986,1.871493
25%,1.313568,0.630698,60208.000000,37830.750000,240.0,320.0,76800.0,0.163752,2.090556
50%,1.460642,0.661378,63670.500000,42066.500000,240.0,320.0,76800.0,0.187242,2.210781
75%,1.614167,0.685466,67915.000000,46488.000000,240.0,320.0,76800.0,0.207417,2.358160
max,1.983090,0.740842,77399.000000,57113.000000,240.0,320.0,76800.0,0.288827,2.687465


In [8]:
""" 4. compress ObjTac (png) """
dataset_path = '/data/ssd/zhaoy/datasets/ObjTac/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/ObjTac/compressed/jxl'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/jxl_ObjTac.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/jxl_ObjTac.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,51.000000,51.000000,51.000000,51.000000,51.0,51.000000,51.000000,51.000000,51.000000
mean,0.382089,0.609238,11784.803922,7689.568627,60.0,903.352941,54201.176471,0.643346,0.578609
std,0.237796,0.136471,7995.848535,5383.188331,0.0,391.917891,23515.073434,0.211885,0.328989
min,0.010645,0.167482,818.000000,137.000000,60.0,331.000000,19860.000000,0.294963,0.063559
25%,0.158008,0.568008,4868.500000,2557.500000,60.0,587.500000,35250.000000,0.516527,0.253969
50%,0.410941,0.647792,10936.000000,6862.000000,60.0,813.000000,48780.000000,0.612408,0.629227
75%,0.569798,0.686846,19075.500000,12658.500000,60.0,1142.000000,68520.000000,0.711425,0.826285
max,0.887760,0.873814,28278.000000,17715.000000,60.0,1802.000000,108120.000000,1.216110,1.246181


In [10]:
""" 5. compress YCB-Slide (png) """
dataset_path = '/data/ssd/zhaoy/datasets/YCB-Slide/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/YCB-Slide/compressed/jxl'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/jxl_YCB-Slide.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/jxl_YCB-Slide.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,10916.000000,10916.000000,10916.000000,10916.000000,10916.0,10916.0,10916.0,10916.000000,10916.000000
mean,1.431337,0.655039,62881.925247,41222.497618,240.0,320.0,76800.0,0.180152,2.183400
std,0.091594,0.014675,2789.584906,2637.897946,0.0,0.0,0.0,0.026014,0.096861
min,1.191424,0.611028,54243.000000,34313.000000,240.0,320.0,76800.0,0.148096,1.883438
25%,1.368125,0.644952,60928.750000,39402.000000,240.0,320.0,76800.0,0.159150,2.115582
50%,1.422934,0.653722,62656.000000,40980.500000,240.0,320.0,76800.0,0.167490,2.175556
75%,1.480911,0.663724,64545.250000,42650.250000,240.0,320.0,76800.0,0.197858,2.241155
max,1.798750,0.715226,73586.000000,51804.000000,240.0,320.0,76800.0,0.276591,2.555069
